# Reproducing "Self-Exploring Language Models for Explainable Link Forecasting on Temporal Graphs via Reinforcement Learning"

This Google Colab notebook aims to reproduce a simplified version of the paper "Self-Exploring Language Models for Explainable Link Forecasting on Temporal Graphs via Reinforcement Learning".

**Paper Abstract Summary:** The paper introduces a novel approach for explainable link forecasting on temporal graphs, leveraging Reinforcement Learning (RL) to fine-tune Large Language Models (LLMs). The LLM processes temporal graph information, predicts future links, and simultaneously generates human-understandable explanations for its predictions. The model uses self-exploring reasoning strategies to enhance the quality and fidelity of explanations.

**This Notebook's Focus:** Due to computational constraints inherent in Google Colab, this notebook implements a significantly simplified version of the proposed methodology. We use a smaller decoder-only LLM (Gemma-2B-IT) and a simplified instruction-tuning approach (LoRA) instead of full RL fine-tuning. We also work with a small subset of the JODIE (Wikipedia) temporal graph dataset and a simplified graph linearization strategy. The goal is to demonstrate the core idea of an LLM taking temporal graph data as input, predicting links, and generating explanations, while acknowledging the architectural and training simplifications.

In [ ]:
# Install all necessary Python packages
!pip install -q torch transformers accelerate bitsandbytes peft torch_geometric tqdm matplotlib scikit-learn

# Ensure protobuf is downgraded for compatibility with some older library versions
# This helps avoid potential conflicts with older dependencies in some environments.
!pip install -q protobuf==3.20.0 

In [ ]:
# Imports and setup
import torch
import random
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from accelerate import Accelerator

from torch_geometric.datasets import JODIE
from torch_geometric.data import TemporalData # JODIE returns this type of data object
from torch.utils.data import Dataset, DataLoader, Subset

from collections import defaultdict
import re
from sklearn.metrics import reciprocal_rank_score

# Set random seeds for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Detect and set up the device (GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Accelerator for mixed precision training
accelerator = Accelerator(mixed_precision="bf16")
print(f"Accelerator mixed precision: {accelerator.mixed_precision}")

In [ ]:
# Load dataset
print("Loading JODIE (Wikipedia) dataset...")
dataset = JODIE(root='/tmp/JODIE', name='wikipedia')
print("Dataset loaded.")

# JODIE returns a single TemporalData object containing all interactions
data = dataset[0]

# Apply the specified subset strategy: first 1000 temporal interactions
# The JODIE dataset is typically sorted chronologically by default.
num_interactions_to_use = 1000
if len(data.src) > num_interactions_to_use:
    data_subset = TemporalData(
        src=data.src[:num_interactions_to_use],
        dst=data.dst[:num_interactions_to_use],
        t=data.t[:num_interactions_to_use],
        msg=data.msg[:num_interactions_to_use]
    )
else:
    data_subset = data

print(f"Using a subset of {len(data_subset.src)} temporal interactions.")

# Chronological train/validation/test split
# 80% train, 10% validation, 10% test
total_events = len(data_subset.src)
train_size = int(0.8 * total_events)
val_size = int(0.1 * total_events)
test_size = total_events - train_size - val_size

train_data = TemporalData(
    src=data_subset.src[:train_size],
    dst=data_subset.dst[:train_size],
    t=data_subset.t[:train_size],
    msg=data_subset.msg[:train_size]
)
val_data = TemporalData(
    src=data_subset.src[train_size : train_size + val_size],
    dst=data_subset.dst[train_size : train_size + val_size],
    t=data_subset.t[train_size : train_size + val_size],
    msg=data_subset.msg[train_size : train_size + val_size]
)
test_data = TemporalData(
    src=data_subset.src[train_size + val_size :],
    dst=data_subset.dst[train_size + val_size :],
    t=data_subset.t[train_size + val_size :],
    msg=data_subset.msg[train_size + val_size :]
)

print(f"Train events: {len(train_data.src)}")
print(f"Validation events: {len(val_data.src)}")
print(f"Test events: {len(test_data.src)}")

# Get all unique node IDs for negative sampling during evaluation
all_nodes = torch.cat([data_subset.src, data_subset.dst]).unique().tolist()
print(f"Total unique nodes in subset: {len(all_nodes)}")

# Build history maps for context generation
# train_history_map: history only from the training split
# val_test_history_map: history from train + validation splits

train_history_map = defaultdict(list)
for i in tqdm(range(len(train_data.src)), desc="Building train history map"):
    src, dst, t, msg = train_data.src[i].item(), train_data.dst[i].item(), train_data.t[i].item(), train_data.msg[i]
    train_history_map[src].append((dst, t, msg.cpu())) # Move msg to CPU if it's on GPU

val_test_history_map = defaultdict(list)
for src_node, events in train_history_map.items():
    val_test_history_map[src_node].extend(events)

for i in tqdm(range(len(val_data.src)), desc="Building val/test history map"):
    src, dst, t, msg = val_data.src[i].item(), val_data.dst[i].item(), val_data.t[i].item(), val_data.msg[i]
    val_test_history_map[src].append((dst, t, msg.cpu())) # Move msg to CPU if it's on GPU

# Sort history maps by timestamp
for src_node in train_history_map:
    train_history_map[src_node].sort(key=lambda x: x[1])
for src_node in val_test_history_map:
    val_test_history_map[src_node].sort(key=lambda x: x[1])

print("Historical interaction maps built.")

In [ ]:
# Preprocess data
class TemporalGraphDataset(Dataset):
    def __init__(self, data: TemporalData, tokenizer, history_map: dict, max_context_events: int = 5, max_seq_len: int = 512):
        self.data = data
        self.tokenizer = tokenizer
        self.history_map = history_map # history map up to the start of this dataset split
        self.max_context_events = max_context_events
        self.max_seq_len = max_seq_len

        self.msg_feature_dim = data.msg.shape[1] if data.msg.dim() > 1 else 0
        if self.msg_feature_dim > 0:
            print(f"Note: Message features (dim {self.msg_feature_dim}) are present but will be simplified in prompts.")

    def __len__(self):
        return len(self.data.src)

    def _get_context_text(self, src_node_id: int, current_time: int) -> str:
        # Retrieve historical interactions for src_node_id before current_time
        historical_events_for_src = []
        
        # Use history from the map provided, which contains events up to the split point
        for dst, t, msg_features in self.history_map.get(src_node_id, []):
            if t < current_time:
                # Simplified message feature string (e.g., just its dimension)
                msg_str = f" (msg_info: dim={msg_features.shape[-1]})" if self.msg_feature_dim > 0 else ""
                historical_events_for_src.append(f"node {src_node_id} interacted with node {dst} at time {t}{msg_str}")
        
        # Take the most recent `max_context_events` events
        historical_events_for_src = historical_events_for_src[-self.max_context_events:]

        if not historical_events_for_src:
            return f"Node {src_node_id} has no recent known interactions."
        else:
            context_text = "; ".join(historical_events_for_src)
            return f"Node {src_node_id}'s recent activity: {context_text}."

    def _generate_explanation_template(self, src_node_id: int, dst_node_id: int) -> str:
        # Simplified dummy explanation for supervised fine-tuning. 
        # In a full RL setup, this would be learned or explicitly guided.
        return f"Based on historical patterns, node {src_node_id} has previously interacted with node {dst_node_id} or similar entities, suggesting a potential connection at this time."

    def __getitem__(self, idx):
        src_node_id = self.data.src[idx].item()
        dst_node_id = self.data.dst[idx].item()
        current_time = self.data.t[idx].item()

        context_text = self._get_context_text(src_node_id, current_time)
        
        # Input prompt for the LLM
        prompt_text = f"Given the context: {context_text} Predict the destination node for node {src_node_id} at time {current_time}. Explain your reasoning.\n"
        
        # Target output for supervised fine-tuning
        explanation = self._generate_explanation_template(src_node_id, dst_node_id)
        target_text = f"TARGET: {dst_node_id} EXPLANATION: {explanation}{self.tokenizer.eos_token}"

        full_text = prompt_text + target_text

        # Tokenize the full sequence
        tokenized_output = self.tokenizer(full_text, return_tensors="pt", 
                                          max_length=self.max_seq_len, padding="max_length", truncation=True)
        input_ids = tokenized_output.input_ids.squeeze(0)
        attention_mask = tokenized_output.attention_mask.squeeze(0)

        labels = input_ids.clone()
        
        # Find the start of the target_text in the full_text by tokenizing the prompt alone
        # This allows masking out the prompt part from the loss calculation
        prompt_tokens_len = len(self.tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids.squeeze(0))
        
        # Set labels for prompt tokens to -100 (ignore in loss calculation)
        if prompt_tokens_len < self.max_seq_len:
             labels[:prompt_tokens_len] = -100 # Mask prompt tokens
        else:
             # If prompt itself is truncated, mark all labels as -100 as the target might be missing
             # or we only have a partial prompt. For this setup, we assume prompts fit.
             labels[:] = -100 

        # Clamp labels to max_seq_len in case of truncation leading to length mismatch
        if len(labels) > self.max_seq_len:
            labels = labels[:self.max_seq_len]
        if len(input_ids) > self.max_seq_len:
            input_ids = input_ids[:self.max_seq_len]
        if len(attention_mask) > self.max_seq_len:
            attention_mask = attention_mask[:self.max_seq_len]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "prompt": prompt_text,
            "target": target_text,
            "src_node_id": src_node_id,
            "dst_node_id": dst_node_id,
            "current_time": current_time
        }

# Load tokenizer for Gemma-2B-IT
print("Loading Gemma tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
tokenizer.pad_token = tokenizer.eos_token # Gemma uses EOS as pad token

# Max sequence length and batch size (adjusted for Colab GPU memory and speed)
MAX_SEQ_LEN = 256 
MAX_CONTEXT_EVENTS = 3 
BATCH_SIZE = 2

train_dataset = TemporalGraphDataset(train_data, tokenizer, train_history_map, MAX_CONTEXT_EVENTS, MAX_SEQ_LEN)
val_dataset = TemporalGraphDataset(val_data, tokenizer, train_history_map, MAX_CONTEXT_EVENTS, MAX_SEQ_LEN) # val uses history up to end of training data
test_dataset = TemporalGraphDataset(test_data, tokenizer, val_test_history_map, MAX_CONTEXT_EVENTS, MAX_SEQ_LEN) # test uses history up to end of val data

print(f"Train dataset samples: {len(train_dataset)}")
print(f"Validation dataset samples: {len(val_dataset)}")
print(f"Test dataset samples: {len(test_dataset)}")

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders created.")

# Example of a preprocessed sample
print("\nExample preprocessed sample:")
sample = train_dataset[0]
print(f"Prompt: {sample['prompt']}")
print(f"Target: {sample['target']}")
print(f"Input IDs (first 20): {sample['input_ids'][:20]}")
print(f"Labels (first 20): {sample['labels'][:20]}")
print(f"Decoded Input (first 50 tokens): {tokenizer.decode(sample['input_ids'][:50])}")
print(f"Decoded Labels (first 50 tokens, -100 masked): {tokenizer.decode(sample['labels'][sample['labels'] != -100][:50])}")

## Model Architecture: Simplified ReaL-TG (Gemma-2B-IT)

The core of our model is a **decoder-only transformer LLM**, specifically `google/gemma-2b-it`. This LLM is adapted to the temporal graph task through a simplified instruction fine-tuning approach.

**Key Components:**
1.  **Pre-trained Decoder-Only Transformer LLM (Gemma-2B-IT):** Serves as the base model, providing strong generative capabilities and understanding of language.
2.  **Graph Linearization Module (Conceptual):** Implemented during data preprocessing, this module converts a snapshot of temporal graph events (e.g., recent interactions involving a source node) into a coherent textual prompt. This prompt describes the graph context and poses the link forecasting question.
3.  **Simplified Instruction Fine-tuning Layer (LoRA adapter):** Instead of full Reinforcement Learning, we employ Low-Rank Adaptation (LoRA) using the `peft` library. This technique significantly reduces the number of trainable parameters, allowing efficient fine-tuning of the LLM on our custom task with limited computational resources. The LoRA adapter is applied to the Gemma-2B-IT model.

**How it works:** The fine-tuned Gemma LLM takes a linearized prompt (e.g., `"Given past events: [event_1], [event_2]... predict the destination for source node X at time T. Explain."`) and generates a sequence containing both the predicted target node ID and a textual explanation for that prediction (e.g., `"TARGET: Y EXPLANATION: Based on node X's past interactions with node Y..."`). The training objective is a standard language modeling loss (CrossEntropyLoss) over this generated sequence.

In [ ]:
# Model Definition

# Quantization configuration for loading Gemma-2B-IT in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Gemma-2B-IT model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16, 
    device_map="auto", 
)

# Prepare model for k-bit training (important for LoRA + quantization)
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"], # Common linear layers in Gemma
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM", 
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

print("LoRA adapted model created.")
model.print_trainable_parameters() # Show number of trainable parameters

# Set generation configuration for the tokenizer's special tokens
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

In [ ]:
# Training Setup

# Optimizer: AdamW
learning_rate = 2e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Loss function: CrossEntropyLoss (for language modeling objective, handled internally by model)
# The model's forward pass returns loss when labels are provided.

# Scheduler: LinearWarmupScheduler
epochs = 1 # As per the reproducibility spec
num_training_steps = epochs * len(train_dataloader)
num_warmup_steps = int(0.1 * num_training_steps) # 10% warmup steps

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

# Prepare everything with accelerator
model, optimizer, train_dataloader, val_dataloader, scheduler = accelerator.prepare(
    model, optimizer, train_dataloader, val_dataloader, scheduler
)

print("Training setup complete: optimizer, scheduler, and accelerator prepared.")

In [ ]:
# Training Loop

train_losses = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs} (Training)")):
        with accelerator.accumulate(model):
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            labels = batch['labels']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            accelerator.backward(loss)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()

    avg_train_loss = total_loss / len(train_dataloader)
    train_losses.append(avg_train_loss)
    print(f"Epoch {epoch+1} finished. Average training loss: {avg_train_loss:.4f}")

    # Validation (optional, but good practice)
    model.eval()
    total_val_loss = 0
    for batch in tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{epochs} (Validation)"):
        with torch.no_grad():
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            labels = batch['labels']
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_val_loss += loss.item()
    
    avg_val_loss = total_val_loss / len(val_dataloader)
    print(f"Validation Loss: {avg_val_loss:.4f}")

print("Training complete.")

In [ ]:
# Evaluation

model.eval()

# Get token IDs for 'Yes' and 'No' to use for scoring candidates.
# Gemma's tokenizer often includes a leading space for common words.
def get_score_tokens(tokenizer):
    # Try to find ' Yes' and ' No' with leading space first
    yes_tokens = tokenizer.encode(" Yes", add_special_tokens=False)
    no_tokens = tokenizer.encode(" No", add_special_tokens=False)

    if len(yes_tokens) == 1 and len(no_tokens) == 1:
        return yes_tokens[0], no_tokens[0]
    
    # Fallback to without leading space if not found (less common for Llama/Gemma style tokenizers)
    yes_tokens = tokenizer.encode("Yes", add_special_tokens=False)
    no_tokens = tokenizer.encode("No", add_special_tokens=False)
    if len(yes_tokens) == 1 and len(no_tokens) == 1:
        return yes_tokens[0], no_tokens[0]

    raise ValueError("Could not reliably find single token IDs for 'Yes' or 'No'.")

try:
    yes_token_id, no_token_id = get_score_tokens(tokenizer)
    print(f"'Yes' token ID: {yes_token_id}, decoded: '{tokenizer.decode(yes_token_id)}'")
    print(f"'No' token ID: {no_token_id}, decoded: '{tokenizer.decode(no_token_id)}'")
except ValueError as e:
    print(f"Error: {e}. Evaluation will proceed with direct generation and parsing, which simplifies MRR/Hits@10 to Hits@1 where ranking is implicit.")
    yes_token_id, no_token_id = None, None # Indicate fallback

all_reciprocal_ranks = []
all_hits_at_10 = 0

# Select a subset of test data for evaluation to meet time constraints
num_test_samples_for_eval = min(100, len(test_dataloader.dataset)) # Evaluate on at most 100 samples
eval_indices = random.sample(range(len(test_dataloader.dataset)), num_test_samples_for_eval)
selected_test_dataset = Subset(test_dataloader.dataset, eval_indices)
eval_dataloader = DataLoader(selected_test_dataset, batch_size=BATCH_SIZE, shuffle=False)
eval_dataloader = accelerator.prepare(eval_dataloader)

# Store example predictions for visualization
example_predictions = []

print(f"Evaluating on {len(selected_test_dataset)} test samples...")

# Keep track of unique nodes for negative sampling
# (Using the `all_nodes` from dataset loading)

for batch_idx, batch in enumerate(tqdm(eval_dataloader, desc="Evaluating")):
    src_node_ids_batch = batch['src_node_id']
    true_dst_node_ids_batch = batch['dst_node_id']
    original_prompts_batch = batch['prompt']
    current_times_batch = batch['current_time']

    for i in range(len(src_node_ids_batch)):
        src = src_node_ids_batch[i].item()
        true_dst = true_dst_node_ids_batch[i].item()
        current_time = current_times_batch[i].item()
        
        # Extract the base context from the original prompt without the question part
        # This ensures the prompt for binary questions is consistent with training context.
        context_start = original_prompts_batch[i].find("Given the context:")
        question_start = original_prompts_batch[i].find("Predict the destination node for node ")
        original_prompt_context = original_prompts_batch[i][context_start:question_start].strip()

        # Create a candidate set: true destination + 9 random distinct negative destinations
        candidates = [true_dst]
        while len(candidates) < 10:
            neg_cand = random.choice(all_nodes)
            if neg_cand not in candidates:
                candidates.append(neg_cand)
        random.shuffle(candidates)
        
        candidate_scores = [] # Stores (score, candidate_id) for ranking

        if yes_token_id is not None and no_token_id is not None:
            # Approach 1: Binary classification for each candidate to get a score
            for cand in candidates:
                binary_question_prompt = f"{original_prompt_context} Predict the destination node for node {src} at time {current_time}. Is it node {cand} ?"
                
                tokenized_question = tokenizer(binary_question_prompt, return_tensors="pt", 
                                               max_length=MAX_SEQ_LEN, truncation=True).to(accelerator.device)
                
                with torch.no_grad():
                    outputs = model(input_ids=tokenized_question.input_ids, attention_mask=tokenized_question.attention_mask)
                    logits = outputs.logits # Logits for the next token prediction
                    
                    # Get logits for the last token position
                    last_token_logits = logits[0, -1, :]
                    
                    # Score based on logit difference between 'Yes' and 'No'
                    score = last_token_logits[yes_token_id] - last_token_logits[no_token_id]
                    candidate_scores.append((score.item(), cand))
            
            # Sort candidates by score in descending order
            candidate_scores.sort(key=lambda x: x[0], reverse=True)
            ranked_candidates = [cand for score, cand in candidate_scores]

            # Calculate MRR and Hits@10
            try:
                rank_of_true_dst = ranked_candidates.index(true_dst) + 1 # 1-based rank
                all_reciprocal_ranks.append(1.0 / rank_of_true_dst)
                if rank_of_true_dst <= 10:
                    all_hits_at_10 += 1
            except ValueError: # Should not happen if true_dst is always in candidates
                all_reciprocal_ranks.append(0.0) # If true_dst somehow not found, count as 0
        
        # Regardless of scoring method, generate a full prediction for display examples
        if len(example_predictions) < 3: # Collect a few examples
            display_prompt = f"{original_prompt_context} Predict the destination node for node {src} at time {current_time}. Explain your reasoning.\n"
            tokenized_display_prompt = tokenizer(display_prompt, return_tensors="pt", 
                                                 max_length=MAX_SEQ_LEN, truncation=True).to(accelerator.device)
            
            # Generate full text including TARGET and EXPLANATION
            generated_ids = model.generate(
                input_ids=tokenized_display_prompt.input_ids,
                attention_mask=tokenized_display_prompt.attention_mask,
                max_new_tokens=64, # Limit new tokens for explanation
                do_sample=True, 
                top_k=50, 
                top_p=0.95,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id
            )
            generated_text = tokenizer.decode(generated_ids[0, tokenized_display_prompt.input_ids.shape[1]:], skip_special_tokens=True)

            # Try to parse predicted target and explanation
            pred_target_node = "N/A (parsing failed)"
            pred_explanation = generated_text
            pred_target_match = re.search(r"TARGET:\s*(\d+)", generated_text)
            if pred_target_match:
                pred_target_node = int(pred_target_match.group(1))
                # Extract explanation after 'EXPLANATION:'
                explanation_match = re.search(r"EXPLANATION:\s*(.*)", generated_text[pred_target_match.end():], re.DOTALL)
                if explanation_match:
                    pred_explanation = explanation_match.group(1).strip()
                else:
                    pred_explanation = generated_text[pred_target_match.end():].strip() # If 'EXPLANATION:' not found, take rest
                
            example_predictions.append({
                "prompt": display_prompt,
                "generated_output": generated_text,
                "predicted_target": pred_target_node,
                "generated_explanation": pred_explanation,
                "ground_truth_target": true_dst
            })


# Calculate final metrics
if len(all_reciprocal_ranks) > 0:
    mrr = np.mean(all_reciprocal_ranks)
    hits_at_10_rate = all_hits_at_10 / len(all_reciprocal_ranks)
else:
    mrr = 0.0
    hits_at_10_rate = 0.0

print(f"\n--- Evaluation Results ({len(all_reciprocal_ranks)} samples) ---")
print(f"Mean Reciprocal Rank (MRR): {mrr:.4f}")
print(f"Hits@10: {hits_at_10_rate:.4f}")

In [ ]:
# Visualization

# 1. Plot training loss curve
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Training Loss')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# 2. Display example predictions with explanations
print("\n--- Example Predictions with Explanations ---")
for i, pred_info in enumerate(example_predictions):
    print(f"\n--- Sample {i+1} ---")
    print(f"Input Prompt:\n{pred_info['prompt']}")
    print(f"LLM Generated Output:\n{pred_info['generated_output']}")
    print(f"Parsed Predicted Target: {pred_info['predicted_target']}")
    print(f"Generated Explanation: {pred_info['generated_explanation']}")
    print(f"Ground Truth Target: {pred_info['ground_truth_target']}")
    print("---------------------")

## Implementation Notes and Simplifications

This notebook demonstrates a simplified reproduction of the paper, subject to various constraints and assumptions, primarily to ensure execution within typical Google Colab GPU limits (e.g., 10-minute execution time).

1.  **Reinforcement Learning vs. Instruction Tuning:** The original paper uses Reinforcement Learning (RL) for fine-tuning. This notebook replaces RL with a **simplified instruction-tuning approach (LoRA SFT)**, where the LLM is trained to generate both the target node ID and an explanation in a supervised manner. This is a significant simplification of the training objective and mechanism.
2.  **LLM Size:** A much smaller LLM, `google/gemma-2b-it`, is used as a stand-in for potentially larger models like Qwen3-4B or more powerful LLMs. This choice is crucial for fitting the model into Colab's GPU memory and execution time.
3.  **Graph Linearization:** The "Graph Linearization Module" is implemented with a **simplified schema**. It focuses on basic conversion of recent temporal interactions into textual prompts without complex graph embedding pre-computation or sophisticated reasoning about graph structures. Message features are noted but simplified in the prompt.
4.  **Dataset Subset:** Only the **first 1000 temporal interactions** from the JODIE (Wikipedia) dataset are used. This expedites data loading and processing but trains on a much smaller and less diverse set of graph data. Furthermore, for evaluation, only ~100 random samples from the test set are used to keep execution within time limits.
5.  **Evaluation Protocol:** The sophisticated "LLM-as-a-Judge" evaluation protocol from the paper is **not directly implemented**. For MRR and Hits@10, a simplified approach is used: the model is prompted with binary questions for a set of 10 candidates (true target + 9 negatives), and scores are derived from the likelihood of generating 'Yes' vs 'No' tokens. This approximates ranking but is less robust than more complex methods.
6.  **Self-Exploration:** The sophisticated "self-exploring reasoning strategies" are implicitly encouraged by prompting the LLM to generate step-by-step explanations, but the explicit RL-driven exploration for improved reasoning is absent.
7.  **Single Epoch Training:** The model is trained for only `1 epoch` to keep execution time minimal. Real-world training would require multiple epochs and careful hyperparameter tuning.
8.  **Explanation Generation:** The target explanations for training are **templated/dummy explanations** rather than real human-generated or sophisticated LLM-generated rationale. This limits the quality of explanations the model can learn to produce.

These simplifications allow for a feasible demonstration of the core concept in a Colab environment, albeit without fully capturing the depth and performance of the original paper's proposed method.